# 第02章 图像处理基础（跟随代码）

> 《OpenCV轻松入门：面向Python》（李立宗）第 2 章的跟随代码。
> **全量收录原书代码**；原 notebook 缺失的例题（例 2.2、2.4–2.6、2.8–2.16、2.19）已按书补全，内容依据站点笔记 ch02.md。
> `cv2.imshow` 需在**本地 Python 环境**运行（弹窗显示图像，网页预览无法显示）；**工作目录须为「数字图像处理-python」根目录**（picture*.jpg 等素材都在这里），否则 imread 返回 None。

In [ ]:
# ===== 通用显示函数（本地运行显示窗口用）=====
import cv2

def cv_show(name, img):
    cv2.imshow(name, img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

## 2.1 图像的基本表示方法

计算机不存"画面"，只存数字：把图像划分成栅格状的小方块（像素，pixel），整幅图就是一个矩阵。三种基本表示：

- **二值图像**（§2.1.1）：只有黑与白，一个比特位（0 或 1）表示一个像素；
- **灰度图像**（§2.1.2）：256 个灰度级，区间 [0,255]，255 纯白、0 纯黑，一个字节存一个像素；
- **彩色图像**（§2.1.3）：三通道各 [0,255]，组合出 256³ = 16 777 216 色；标准顺序 RGB，**OpenCV 按 B→G→R 存储**。

本节为概念铺垫，无代码；从 §2.2 开始动手处理像素（例 2.1–2.6）。

## 2.2 像素处理

图像就是 numpy 数组，"处理像素"就是"给数组元素赋值"：灰度图 `img[行, 列]`、彩色图 `img[行, 列, 通道]`，行列都从 0 开始、第 0 行第 0 列在左上角。本节含例 2.1–2.6。

### 例 2.1 · 生成并修改 8×8 黑图

`np.zeros((8,8), dtype=np.uint8)` 就是一幅纯黑图像；`img[0,3]` 读第 0 行第 3 列，赋 255 后窗口顶边多一个白点。

注意别忘 `dtype=np.uint8`：默认 float64 会让 imshow 显示异常。

In [ ]:
# ===== 2.2 例 2.1：8×8 全零数组 = 一幅纯黑图像 =====
import cv2
import numpy as np

img = np.zeros((8, 8), dtype=np.uint8)    # 8 行 8 列，全 0；uint8 保证每像素 0~255
print("img=\n", img)                      # 打印就是 8×8 的零矩阵
cv2.imshow("one", img)                    # 窗口只有 8×8 像素，很小，纯黑

print("读取像素点img[0,3]=", img[0, 3])    # 第 0 行第 3 列 → 0
img[0, 3] = 255                           # 把该点置为 255（白）
print("修改后img=\n", img)
print("读取修改后像素点img[0,3]=", img[0, 3])

cv2.imshow("two", img)                    # 顶边中间多出一个白点
cv2.waitKey()
cv2.destroyAllWindows()

### 例 2.2 · 把灰度图的一块区域刷白

双循环遍历第 10~99 行 × 第 80~99 列，逐像素赋 255——灰度数组没有通道维，一个赋值改完一个像素。改后左上偏左位置出现 90 行 × 20 列的白色竖条。

In [ ]:
# ===== 2.2 例 2.2：把一块矩形区域刷成白色 =====
import cv2

img = cv2.imread("picture1.jpg", 0)     # flags=0：直接读成单通道灰度图（原书 lena.bmp）
cv2.imshow("before", img)

for i in range(10, 100):                # 第 10~99 行，共 90 行
    for j in range(80, 100):            # 第 80~99 列，共 20 列
        img[i, j] = 255                 # 交叉区域全刷成白

cv2.imshow("after", img)
cv2.waitKey()
cv2.destroyAllWindows()

### 例 2.3 · 把三个通道分别点亮，看颜色

`np.zeros((300,300,3), dtype=np.uint8)` 后把第 0/1/2 通道分别拉满 255，得到纯蓝、纯绿、纯红三幅图——`img[:,:,0]` 的 `:` 表示该维全取。

OpenCV 通道顺序是 **B→G→R**：第 0 个通道是蓝、第 2 个才是红。

In [ ]:
# ===== 2.2 例 2.3：把三个通道分别点亮，看颜色 =====
import numpy as np
import cv2

# ---------- 蓝色通道 ----------
blue = np.zeros((300, 300, 3), dtype=np.uint8)   # 300×300，3 通道，全 0
blue[:, :, 0] = 255                              # 第 0 个通道（B）全置 255
print("blue[0,0]=", blue[0, 0])                   # [255   0   0]：B=255 → 蓝
cv2.imshow("blue", blue)

# ---------- 绿色通道 ----------
green = np.zeros((300, 300, 3), dtype=np.uint8)
green[:, :, 1] = 255                             # 第 1 个通道（G）全置 255
print("green[0,0]=", green[0, 0])                 # [  0 255   0]：G=255 → 绿
cv2.imshow("green", green)

# ---------- 红色通道 ----------
red = np.zeros((300, 300, 3), dtype=np.uint8)
red[:, :, 2] = 255                               # 第 2 个通道（R）全置 255
print("red[0,0]=", red[0, 0])                     # [  0   0 255]：R=255 → 红
cv2.imshow("red", red)

cv2.waitKey()
cv2.destroyAllWindows()

### 例 2.4 · 左中右三条竖条分别点亮 B、G、R

`img[:, 0:100, 0]=255` 这类切片把"所有行、某段列、某个通道"的区域整体赋值，从左到右得到蓝、绿、红三条竖条——这是 2.4 节 ROI 与 2.5 节通道操作的提前预演。

In [ ]:
# ===== 2.2 例 2.4：左中右三条竖条分别点亮 B、G、R =====
import numpy as np
import cv2

img = np.zeros((300, 300, 3), dtype=np.uint8)
img[:, 0:100, 0] = 255       # 第 0~99 列：B 通道拉满
img[:, 100:200, 1] = 255     # 第 100~199 列：G 通道拉满
img[:, 200:300, 2] = 255     # 第 200~299 列：R 通道拉满

print("img[0,0]=", img[0, 0])        # [255   0   0]
print("img[0,150]=", img[0, 150])    # [  0 255   0]
print("img[0,250]=", img[0, 250])    # [  0   0 255]
cv2.imshow("img", img)
cv2.waitKey()
cv2.destroyAllWindows()

### 例 2.5 · 小数组上练彩色索引

2 行 4 列 3 通道的小数组，逐条演示彩色图读写：`img[0,3]` 读整个 BGR 三元组；`img[0,3]=255` 三通道一起改白；`img[0,0]=[66,77,88]` 三个分量分别赋值（列表长度必须为 3）；`img[1,1,1]=3` 只改单个通道。

In [ ]:
# ===== 2.2 例 2.5：小数组上练彩色索引 =====
import numpy as np

img = np.zeros((2, 4, 3), dtype=np.uint8)     # 2 行 4 列 3 通道，全 0
print("img=\n", img)
print("读取像素点img[0,3]=", img[0, 3])        # 第 0 行第 3 列的 BGR 三元组
print("读取像素点img[1,2,2]=", img[1, 2, 2])   # 第 1 行第 2 列第 2 通道（R）

img[0, 3] = 255            # 第 0 行第 3 列：三通道一起置 255 → 白
img[0, 0] = [66, 77, 88]   # 第 0 行第 0 列：分别置 B=66、G=77、R=88
img[1, 1, 1] = 3           # 第 1 行第 1 列 G 通道 → 3
img[1, 2, 2] = 4           # 第 1 行第 2 列 R 通道 → 4
img[0, 2, 0] = 5           # 第 0 行第 2 列 B 通道 → 5

print("修改后img\n", img)
print("读取修改后像素点img[1,2,2]=", img[1, 2, 2])

### 例 2.6 · 在真实彩色图上改三个区域

左上角拼出"白—灰—黑"三条 50 行 × 100 列的色带，三种赋值写法对比：三层循环逐通道改（最笨最直观）、`img[i,j]=[128,128,128]` 三分量同时赋值、`img[i,j]=0` 标量广播到三通道。

拓展：区域 1 的三层循环可一行切片代替——`img[0:50, 0:100] = 255`，速度差两个数量级，这正是 2.4 节 ROI 的内容。

In [ ]:
# ===== 2.2 例 2.6：左上角刷白、刷灰、刷黑三块 =====
import cv2

img = cv2.imread("picture1.jpg")           # 默认读成 3 通道 BGR（原书 lenacolor.png）
cv2.imshow("before", img)
print("访问img[0,0]=", img[0, 0])           # 第 0 行第 0 列的 BGR 三元组
print("访问img[0,0,0]=", img[0, 0, 0])      # B
print("访问img[0,0,1]=", img[0, 0, 1])      # G
print("访问img[0,0,2]=", img[0, 0, 2])      # R
print("访问img[50,0]=", img[50, 0])
print("访问img[100,0]=", img[100, 0])

# 区域 1：第 0~49 行 × 第 0~99 列 → 白
for i in range(0, 50):
    for j in range(0, 100):
        for k in range(0, 3):
            img[i, j, k] = 255

# 区域 2：第 50~99 行 × 第 0~99 列 → 灰
for i in range(50, 100):
    for j in range(0, 100):
        img[i, j] = [128, 128, 128]

# 区域 3：第 100~149 行 × 第 0~99 列 → 黑
for i in range(100, 150):
    for j in range(0, 100):
        img[i, j] = 0

cv2.imshow("after", img)
print("修改后img[0,0]=", img[0, 0])
print("修改后img[0,0,0]=", img[0, 0, 0])
print("修改后img[0,0,1]=", img[0, 0, 1])
print("修改后img[0,0,2]=", img[0, 0, 2])
print("修改后img[50,0]=", img[50, 0])
print("修改后img[100,0]=", img[100, 0])
cv2.waitKey()
cv2.destroyAllWindows()

## 2.3 使用 numpy.array 访问像素

`item()` 与 `itemset()` 两个方法读写单个像素值，书上说它们"经过优化处理，效率比索引高、可读性更好"：

```text
item(行, 列[, 通道])            # 读单个像素值
itemset(索引值, 新值)           # 写单个像素值
```

**勘误：`itemset` 在 NumPy 2.0 已被移除**，运行会直接抛 `AttributeError`，官方替代写法就是索引赋值 `img[索引] = 新值`（两者等价）；`item()` 仍可正常使用。本节例题按新写法给出，注释里标出原书写法。

### 例 2.7 · item 读、索引写

5×5 随机数组模拟灰度图：`img.item(3,2)` 读第 3 行第 2 列；写回用 `img[3,2]=255`（原书 `img.itemset((3,2),255)` 在 NumPy 2.0 已不可用）。

In [ ]:
# ===== 2.3 例 2.7：item 读、索引写 =====
import numpy as np

img = np.random.randint(10, 99, size=[5, 5], dtype=np.uint8)  # 10~98 的随机整数
print("img=\n", img)
print("读取像素点img.item(3,2)=", img.item(3, 2))   # 第 3 行第 2 列
img[3, 2] = 255                                     # 原书：img.itemset((3,2),255)
print("修改后img=\n", img)
print("修改后像素点img.item(3,2)=", img.item(3, 2))

### 例 2.8 · 生成一幅 256×256 随机灰度图

`np.random.randint(0, 256, ...)` 画出来就是"电视雪花"噪点。注意 `randint` 的上界取不到：写 256 才能取到 255，写成 255 最大只有 254。

In [ ]:
# ===== 2.3 例 2.8：全随机灰度图 =====
import numpy as np
import cv2

img = np.random.randint(0, 256, size=[256, 256], dtype=np.uint8)  # 0~255 全覆盖
cv2.imshow("demo", img)          # 看起来就是电视雪花噪点
cv2.waitKey()
cv2.destroyAllWindows()

### 例 2.9 · 真实灰度图上读改单个点 + 刷白一块区域

`item()` 读点、索引赋值改点，双循环刷区域；原书 `itemset` 写法在注释里标出。

In [ ]:
# ===== 2.3 例 2.9：单个点 + 整块区域 =====
import cv2

img = cv2.imread("picture1.jpg", 0)                 # 灰度读入（原书 lena.bmp）
print("读取像素点img.item(3,2)=", img.item(3, 2))
img[3, 2] = 255                                     # 原书：img.itemset((3,2),255)
print("修改后像素点img.item(3,2)=", img.item(3, 2))

cv2.imshow("before", img)
for i in range(10, 100):                            # 第 10~99 行
    for j in range(80, 100):                        # 第 80~99 列
        img[i, j] = 255                             # 原书：img.itemset((i,j),255)
cv2.imshow("after", img)
cv2.waitKey()
cv2.destroyAllWindows()

### 例 2.10 · 彩色图的 item 必须给三个索引

`img.item(1,2,0)` 读第 1 行第 2 列 B 通道。彩色图访问必须同时给行、列、通道三个索引，少给一维报 `ValueError: incorrect number of indices for array`。

In [ ]:
# ===== 2.3 例 2.10：彩色图的 item 必须给三个索引 =====
import numpy as np

img = np.random.randint(10, 99, size=[2, 4, 3], dtype=np.uint8)
print("img=\n", img)

print("读取像素点img[1,2,0]=", img.item(1, 2, 0))   # 第 1 行第 2 列 B 通道
print("读取像素点img[0,2,1]=", img.item(0, 2, 1))   # 第 0 行第 2 列 G 通道
print("读取像素点img[1,0,2]=", img.item(1, 0, 2))   # 第 1 行第 0 列 R 通道

img[1, 2, 0] = 255                                  # 原书：img.itemset((1,2,0),255)
img[0, 2, 1] = 255
img[1, 0, 2] = 255

print("修改后img=\n", img)
print("修改后像素点img[1,2,0]=", img.item(1, 2, 0))
print("修改后像素点img[0,2,1]=", img.item(0, 2, 1))
print("修改后像素点img[1,0,2]=", img.item(1, 0, 2))

### 例 2.11 · 生成 256×256×3 的随机彩色图

三通道各自随机，显示为"彩色雪花"。

In [ ]:
# ===== 2.3 例 2.11：随机彩色噪点图 =====
import cv2
import numpy as np

img = np.random.randint(0, 256, size=[256, 256, 3], dtype=np.uint8)
cv2.imshow("demo", img)          # 三通道各自随机，显示为彩色雪花
cv2.waitKey()
cv2.destroyAllWindows()

### 例 2.12 · 真实彩色图上读三点、刷白一块

`img.item(0,0,0/1/2)` 读 B/G/R 三个分量，三层循环把左上角区域刷白。原书此处漏了 waitKey / destroyAllWindows，窗口会一闪而过，笔记版已补上。

In [ ]:
# ===== 2.3 例 2.12：彩色图上读三点、刷白一块 =====
import cv2

img = cv2.imread("picture1.jpg")               # 原书 lenacolor.png
cv2.imshow("before", img)

print("访问img.item(0,0,0)=", img.item(0, 0, 0))   # B
print("访问img.item(0,0,1)=", img.item(0, 0, 1))   # G
print("访问img.item(0,0,2)=", img.item(0, 0, 2))   # R

for i in range(0, 50):                         # 第 0~49 行
    for j in range(0, 100):                    # 第 0~99 列
        for k in range(0, 3):                  # 三个通道
            img[i, j, k] = 255                 # 原书：img.itemset((i,j,k),255)

cv2.imshow("after", img)
print("修改后img.item(0,0,0)=", img.item(0, 0, 0))
print("修改后img.item(0,0,1)=", img.item(0, 0, 1))
print("修改后img.item(0,0,2)=", img.item(0, 0, 2))

cv2.waitKey()                # 原书此处漏了这两行，窗口会一闪而过
cv2.destroyAllWindows()

## 2.4 感兴趣区域（ROI）

感兴趣区域（Region of Interest，ROI）用**切片**表示：`img[行起:行止, 列起:列止]`，行在前、列在后，`:` 表示全取。切片**左闭右开**：`img[200:400, 200:400]` 取第 200~399 行、第 200~399 列，正好 200×200。

圈定 ROI 后可当整体操作：赋给变量、整块覆盖、搬到别处。注意切片是**视图**（与原图共享内存），改它等于改原图；要独立副本用 `.copy()`。

In [ ]:
# ===== 2.4 ROI 复制：截一块贴到同图另一处 =====
import cv2

img = cv2.imread("picture1.jpg")
a = img[200:400, 200:400]      # 取出 ROI（此时 a 与原图共享内存）
img[200:400, 600:800] = a      # 贴到右侧
cv_show("copy", img)

### 例 2.13 · 截出 ROI 并显示

`face = a[220:400, 250:350]` 截出 180 行 × 100 列——行数 = 400−220、列数 = 350−250，左闭右开一眼算出。

In [ ]:
# ===== 2.4 例 2.13：按坐标截一块 =====
import cv2

a = cv2.imread("picture1.jpg", cv2.IMREAD_UNCHANGED)   # 保持原格式读入
face = a[220:400, 250:350]                             # 第 220~399 行 × 第 250~349 列
print(face.shape)                                      # (180, 100, 3)：180 行 100 列

cv2.imshow("original", a)
cv2.imshow("face", face)
cv2.waitKey()
cv2.destroyAllWindows()

### 例 2.14 · 随机打码

用与 ROI 同形状的随机数组整块覆盖：`face = np.random.randint(0, 256, (180, 100, 3))`。**形状必须一致**，不匹配会被 NumPy 拒绝（`could not broadcast input array from shape ... into shape ...`）。

`randint` 不给 dtype 时返回 32 位整型，赋进 uint8 自动转换但白占 4 倍内存，加 `dtype=np.uint8` 更干净。

In [ ]:
# ===== 2.4 例 2.14：用随机数组覆盖 ROI =====
import cv2
import numpy as np

a = cv2.imread("picture1.jpg", cv2.IMREAD_UNCHANGED)
cv2.imshow("original", a)

face = np.random.randint(0, 256, (180, 100, 3))   # 与 ROI 同形状的随机数组（原书未指定 dtype）
a[220:400, 250:350] = face                         # 整块覆盖 = 打码

cv2.imshow("result", a)
cv2.waitKey()
cv2.destroyAllWindows()

### 例 2.15 · 把一幅图的 ROI 贴到另一幅图

两幅图尺寸不同没关系，只要**目标区域与源 ROI 形状相同**就能贴：`dst[160:340, 200:300] = src[220:400, 250:350]`。跨图搬运本质还是切片赋值，OpenCV 没有任何额外的拷贝函数。

In [ ]:
# ===== 2.4 例 2.15：跨图搬运 ROI =====
import cv2

src = cv2.imread("picture1.jpg", cv2.IMREAD_UNCHANGED)    # 原书 lena512.bmp
dst = cv2.imread("picture12.jpg", cv2.IMREAD_UNCHANGED)   # 原书 dollar.bmp
cv2.imshow("src", src)
cv2.imshow("dst", dst)

face = src[220:400, 250:350]         # 源图 ROI：180 行 100 列
dst[160:340, 200:300] = face          # 目标区域必须也是 180 行 100 列

cv2.imshow("result", dst)
cv2.waitKey()
cv2.destroyAllWindows()

## 2.5 通道操作

RGB 图像由 R、G、B 三个通道构成，OpenCV 按 B→G→R 存储。通道可以拆开单独处理（§2.5.1），也可以重新合并（§2.5.2）：拆分用索引 `img[:,:,0/1/2]` 或 `cv2.split()`，合并用 `cv2.merge()`。

注意：索引拆出的是**视图**，改它就是改原图；`split` 返回顺序固定是 B、G、R。

### 例 2.16 · 索引拆通道 + 逐通道清零

`b = lena[:,:,0]` 等三条语句拆出三个通道（都是二维数组，可当灰度图显示）；再逐通道清零看颜色变化：B 清零画面偏黄（只剩 R+G），再清 G 只剩红色调。

`b/g/r` 是视图：`lena[:,:,0]=0` 时 `b` 跟着一起变。

In [ ]:
# ===== 2.5.1 例 2.16：索引拆通道 + 逐通道置 0 =====
import cv2

lena = cv2.imread("picture1.jpg")        # 原书 lenacolor.png
cv2.imshow("lena1", lena)

b = lena[:, :, 0]                        # B 通道（视图）
g = lena[:, :, 1]                        # G 通道
r = lena[:, :, 2]                        # R 通道
cv2.imshow("b", b)
cv2.imshow("g", g)
cv2.imshow("r", r)

lena[:, :, 0] = 0                        # B 清零 → 画面偏黄（只剩 R+G）
cv2.imshow("lenab0", lena)

lena[:, :, 1] = 0                        # 再清 G → 画面只剩红色分量
cv2.imshow("lenab0g0", lena)

cv2.waitKey()
cv2.destroyAllWindows()

### 例 2.17 · 用 cv2.split() 拆通道

`b, g, r = cv2.split(lena)` 一次拆出全部通道，等价于逐条 `cv2.split(img)[0/1/2]`。返回值顺序固定是 B、G、R，不会因为变量名叫什么就自动对应。

In [ ]:
# ===== 2.5.1 例 2.17：cv2.split 拆三通道 =====
import cv2

lena = cv2.imread("picture1.jpg")
b, g, r = cv2.split(lena)      # 依次得到 B、G、R 三个二维数组
print(b.shape, g.shape, r.shape)   # (1080, 1920) 三个都是单通道

cv2.imshow("B", b)
cv2.imshow("G", g)
cv2.imshow("R", r)
cv2.waitKey()
cv2.destroyAllWindows()

### 例 2.18 · merge 的顺序决定颜色

`cv2.merge([b, g, r])` 把三张单通道图拼回彩色图，**列表顺序决定输出布局**：`[b,g,r]` 与原图逐像素相同；`[r,g,b]` 红蓝互换（红衣服变蓝、蓝天变红）——这正是第 1 章"BGR 不是 RGB"那个坑的另一种触发方式。

In [ ]:
# ===== 2.5.2 例 2.18：merge 的顺序决定颜色 =====
import cv2

lena = cv2.imread("picture1.jpg")
b, g, r = cv2.split(lena)          # 拆出三个通道

bgr = cv2.merge([b, g, r])         # B→G→R：与原图完全一致
rgb = cv2.merge([r, g, b])         # R→G→B：红蓝互换

cv2.imshow("lena", lena)
cv2.imshow("bgr", bgr)
cv2.imshow("rgb", rgb)
cv2.waitKey()
cv2.destroyAllWindows()

print((bgr == lena).all())         # True：bgr 与 lena 像素级完全相同

## 2.6 获取图像属性

| 属性 | 含义 | 彩色图 | 灰度图 |
| --- | --- | --- | --- |
| `shape` | 行数、列数（、通道数） | (行, 列, 3) | (行, 列) |
| `size` | 像素（元素）总数 | 行 × 列 × 3 | 行 × 列 |
| `dtype` | 数据类型 | uint8 | uint8 |

`shape` 最实用：**返回值长度就是判别依据**——有第三维是彩色图，只有两维是灰度/二值图。`size` 数的是元素个数，不是字节数（字节数看 `nbytes`）。

In [ ]:
# ===== 2.6 例 2.19：shape / size / dtype =====
import cv2

gray = cv2.imread("picture1.jpg", 0)        # 原书 lena.bmp（256×256）
color = cv2.imread("picture1.jpg")          # 原书 lenacolor.png（512×512×3）

print("图像gray 属性：")
print("gray.shape=", gray.shape)            # (1080, 1920)
print("gray.size=", gray.size)              # 2073600 = 1080×1920
print("gray.dtype=", gray.dtype)            # uint8

print("图像color 属性：")
print("color.shape=", color.shape)          # (1080, 1920, 3)
print("color.size=", color.size)            # 6220800 = 1080×1920×3
print("color.dtype=", color.dtype)          # uint8